In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # Hugging Face login

!pip install -q "omnifall[video]"
!ffmpeg -version | head -1
OUT = "/kaggle/working/omnifall"
DL = "/tmp/omnifall_dl"
os.makedirs(OUT, exist_ok=True)

In [ ]:
for d in ["mcfd", "caucafall", "le2i", "OOPS"]:
    print(f"\n===== {d} =====", flush=True)
    os.makedirs(f"{OUT}/{d}/video", exist_ok=True)
    !omnifall prepare {d} --yes --download-dir {DL} --video-dir {OUT}/{d}/video
    !rm -rf {DL}
    !du -sh {OUT}/{d}

In [ ]:
for d in ["mcfd", "caucafall", "le2i"]:
    print(f"\n===== {d} =====", flush=True)
    os.makedirs(f"{OUT}/{d}/video", exist_ok=True)
    !omnifall prepare {d} --yes --download-dir {DL} --video-dir {OUT}/{d}/video
    !rm -rf {DL}
    !du -sh {OUT}/{d}

In [ ]:
import requests
from pathlib import Path
from omnifall._oops import load_mapping, extract_oops

# the official OOPS link returns 404; the same archive is mirrored on Hugging Face
candidates = [
    "https://huggingface.co/datasets/pengxiang/oops/resolve/main/video_and_anns.tar.gz",
    "https://huggingface.co/datasets/pengxiang/oops/resolve/main/oops/video_and_anns.tar.gz",
]
src = None
for u in candidates:
    r = requests.head(u, allow_redirects=True, timeout=60)
    print(r.status_code, u)
    if r.status_code == 200:
        src = u
        break
assert src, "Mirror not reachable - send this output to the team"

out_dir = Path(f"{OUT}/OOPS/video")
out_dir.mkdir(parents=True, exist_ok=True)
mapping = load_mapping()
print(f"\n===== OOPS =====\n{len(mapping)} fall clips wanted; streaming ~48 GB, keeping ~2.6 GB (30-60 min)", flush=True)
try:
    n = extract_oops(src, mapping, out_dir, resume=True)
    print(f"OOPS: {n} videos written")
except Exception as e:
    print("OOPS FAILED:", e)
!du -sh {OUT}/OOPS

In [ ]:
for d in ["mcfd", "caucafall", "le2i", "OOPS"]:
    !omnifall verify {d} --video-dir {OUT}/{d}/video
    !echo "{d}: $(find {OUT}/{d} -name '*.mp4' | wc -l) videos"
!du -sh {OUT}